In [ ]:
import cobrabox as cb

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
dtset = cb.dataset("dummy_noise")
print(dtset)

In [ ]:
current_subject = dtset[4]
print(current_subject)

In [ ]:
current_subject.extra

In [ ]:
print(f"Sampling rate: {current_subject.sampling_rate}")
print(f"Seizure start (sec): {current_subject.extra['Seizure start (sec)']}")
print(f"Seizure start (samples): {current_subject.extra['Seizure start (samples)']}")

In [ ]:
labels_SOZ = current_subject.extra["SOZ"]
sz_start = current_subject.extra["Seizure start (samples)"]
sz_end = current_subject.extra["Seizure end (samples)"]

N_channels = current_subject.data.shape[0]
time_vector = list(range(current_subject.data.shape[1]))

print(f"Number of channels: {N_channels}")
print(f"Number of time points: {len(time_vector)}")

In [ ]:
eeg_signal = current_subject.data.T

fig = go.Figure()

offset = 10
for ch in range(N_channels):
    if labels_SOZ[ch]:
        use_color = "red"
        legend_group = "SOZ"
    else:
        use_color = "blue"
        legend_group = "Healthy"

    fig.add_trace(
        go.Scatter(
            x=time_vector[:],
            y=eeg_signal[:, ch] + ch * offset,
            mode="lines",
            name=f"Ch {ch}",
            line=dict(color=use_color),
            hovertemplate=f"Ch {ch}<br>Time: %{{x:.2f}}s<br>Amplitude: %{{y:.2f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Simulated LFP",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (a.u.)",
    hovermode="x unified",
    # height=600,
    # width=1200
)

fig.show()

In [ ]:
feat = cb.feature.LineLength().apply(current_subject)
print(feat)

In [ ]:
# Extract feature data and create labels
feat_data = feat.data.values.flatten()
soz_labels = labels_SOZ.copy()
soz_labels
# Create DataFrame for easier plotting
plot_data = pd.DataFrame(
    {
        "Feature Value": feat_data,
        "Channel Type": ["SOZ" if label else "Healthy" for label in soz_labels],
    }
)

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
sns.violinplot(
    data=plot_data,
    x="Channel Type",
    y="Feature Value",
    hue="Channel Type",
    ax=axes[0],
    palette=["#d62728", "#1f77b4"],
    legend=False,
)
axes[0].set_title(
    "Distribution of Feature Values by Channel Type (Violin Plot)", fontsize=12, fontweight="bold"
)
axes[0].set_ylabel("Feature Value")
axes[0].set_xlabel("")

# KDE plot
sns.kdeplot(
    data=plot_data[plot_data["Channel Type"] == "SOZ"],
    x="Feature Value",
    ax=axes[1],
    label="SOZ",
    color="red",
    fill=True,
    alpha=0.5,
)
sns.kdeplot(
    data=plot_data[plot_data["Channel Type"] == "Healthy"],
    x="Feature Value",
    ax=axes[1],
    label="Healthy",
    color="blue",
    fill=True,
    alpha=0.5,
)
axes[1].set_title("Distribution of Feature Values (KDE Plot)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Feature Value")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("\nDistribution Statistics:")
print(plot_data.groupby("Channel Type")["Feature Value"].describe())

In [ ]:
feature = cb.feature.LineLength()
concat_result = (
    cb.feature.SlidingWindow(window_size=50, step_size=25) | feature | cb.feature.ConcatAggregate()
).apply(current_subject)

print("=== ConcatAggregate ===")
print(f"  Shape:   {dict(concat_result.data.sizes)}")
print(f"  Dims:    {concat_result.data.dims}")
print(f"  History: {concat_result.history}")
print()

# Inspect individual windows from ConcatAggregate
n_windows = concat_result.data.sizes["window"]
print(f"Number of windows retained: {n_windows}")
print(f"Window 0 values (first channel): {concat_result.data.isel(window=0).values[:5]} ...")
print(f"Window 1 values (first channel): {concat_result.data.isel(window=1).values[:5]} ...")

In [ ]:
import numpy as np

moving_features = np.array(
    [concat_result.data.isel(window=iW).values[:] for iW in range(n_windows)]
)[:, :, 0]
sns.heatmap(moving_features.T, cmap="plasma", cbar_kws={"label": "Feature Value"})

In [ ]:
LEN_TS, N = moving_features.shape[0], moving_features.shape[1]
tvec = np.arange(LEN_TS) * 0.5  # Assuming step_size=25 samples at 50 Hz sampling rate

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

offset = -20
for ch in range(N):
    if labels_SOZ[ch]:
        use_color = "red"
        legend_group = "SOZ"
    else:
        use_color = "blue"
        legend_group = "Healthy"

    fig.add_trace(
        go.Scatter(
            x=tvec[:],
            y=moving_features[:, ch],  # + ch*offset,
            mode="lines",
            name=f"Ch {ch}",
            line=dict(color=use_color),
            hovertemplate=f"Ch {ch}<br>Time: %{{x:.2f}}s<br>Amplitude: %{{y:.2f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Moving line length feature across channels",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (a.u.)",
    hovermode="x unified",
    height=600,
    width=1200,
)

fig.show()